# CNN + BiLSTM + Attention (Experiment)

**Challenge:** AN2DL 2024 — Pirate Pain Classification  
**Approach:** 1D CNN feature extractor + Bidirectional LSTM with a Tanh attention mechanism,  
categorical embeddings (`nn.Embedding`), thresholding of near-zero noisy joints, and weighted cross-entropy.  
**Result:** val F1 ≈ 0.92 — competitive but below the best BiDir LSTM+Conv1D model.

### Pipeline
1. EDA: null-value checks, shape inspection, joint-distribution boxplots
2. Min-max normalisation on joint columns (fitted on train split only)
3. Threshold denoising: joints 21-25 (near-zero noise) zeroed below 0.1
4. Sliding-window sequence builder (window=20, stride=5)
5. `nn.Embedding` for `n_legs`, `n_hands`, `n_eyes`
6. `Conv1D(in=34, out=64, kernel=3)` → BiLSTM(128) → Tanh Attention → Linear(3)
7. Weighted `CrossEntropyLoss`, AdamW, early stopping (patience=50)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import random
import copy
import os
import re
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from statsmodels.tsa.stattools import acf


In [ ]:
# TensorBoard is available but optional; comment out if not installed
try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:
    SummaryWriter = None


In [ ]:
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
SEED = 42 # To guarantee reproducibility
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
#Set environment variables

os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

In [ ]:
#Suppress warnings

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

In [ ]:
# Set torch options

logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models


# Use a GPU if available, or use a CPU instead (N.B. GPUs are made available from kaggle)
if torch.cuda.is_available():     
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

# Check torch version 
print(f"PyTorch version: {torch.__version__}")


# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

# Data Loading 

In [ ]:
#Data Loading

X = pd.read_csv('/kaggle/input/pirate-pain/pirate_pain_train.csv')
y = pd.read_csv('/kaggle/input/pirate-pain/pirate_pain_train_labels.csv')
X_test = pd.read_csv('/kaggle/input/pirate-pain/pirate_pain_test.csv')

# Data Exploration and Data Analysis

In [ ]:
print("Count of null values in features:")
X.isnull().sum()

In [ ]:
print("Count of null values in targets:")
y.isnull().sum()

In [ ]:
print("Count of null values in test targets:")
X_test.isnull().sum()

In [ ]:
print(X.shape)
print(y.shape)
print(X_test.shape)

In [ ]:
print(X.head(5))

In [ ]:
X.info()

In [ ]:
X.describe() # General stats about the dataset

In [ ]:
print("--- Visualizing Joint Distributions ---")

# Get all joint columns
joint_cols = [col for col in X.columns if re.match(r'joint_\\d{2}', col)]
# Exclude the constant joint_30 for a cleaner plot
joint_cols_to_plot = [col for col in joint_cols if col != 'joint_30']

# Due to the huge difference in scales, we'll plot in two groups
# Group 1: More active joints (based on .describe())
active_joints = ['joint_00', 'joint_01', 'joint_02', 'joint_03', 'joint_04', 
                 'joint_05', 'joint_06', 'joint_07', 'joint_08', 'joint_09', 
                 'joint_10', 'joint_11', 'joint_12', 'joint_13', 'joint_14', 
                 'joint_15', 'joint_16', 'joint_17', 'joint_18', 'joint_19',
                 'joint_20', 'joint_26', 'joint_27', 'joint_28', 'joint_29']

# Group 2: Less active / noisy joints
noisy_joints = ['joint_21', 'joint_22', 'joint_23', 'joint_24', 'joint_25']

# Plot Group 1
plt.figure(figsize=(20, 10))
sns.boxplot(data=X[active_joints])
plt.title('Boxplots for "Active" Joints')
plt.xticks(rotation=45)
plt.show()

# Plot Group 2
plt.figure(figsize=(20, 10))
sns.boxplot(data=X[noisy_joints])
plt.title('Boxplots for "Noisy" Joints')
plt.xticks(rotation=45)
plt.show()

# Plot Group 2 with a zoom-in (log scale) to see the distribution
plt.figure(figsize=(20, 10))
sns.boxplot(data=X[noisy_joints])
plt.title('Boxplots for "Noisy" Joints (Log Scale Y-Axis)')
plt.yscale('log')
plt.xticks(rotation=45)
plt.show()

In [ ]:
#How many time stamps for each sample?

counts = X.groupby("sample_index")["time"].count()
all(counts == 160) # True, exactly 160 timestamps each

In [ ]:
# How many samples?
samples = X['sample_index'].unique()
n_samples = len(samples) 
print(n_samples) # 661

In [ ]:
random.shuffle(samples)

In [ ]:
N_VAL_SAMPLES = 60 # Number of samples used for validation (tunable parameter)

n_train_samples = len(samples) - N_VAL_SAMPLES

print(n_train_samples)

# Data Preprocessing

In [ ]:
# Split the shuffled samples id into training and validation
train_samples = samples[:n_train_samples]
val_samples = samples[n_train_samples:n_train_samples + N_VAL_SAMPLES]

In [ ]:
# Split the dataset into training and validation sets based on samples ids
X_train = X[X['sample_index'].isin(train_samples)]
X_val = X[X['sample_index'].isin(val_samples)]

y_train = y[y['sample_index'].isin(train_samples)]
y_val = y[y['sample_index'].isin(val_samples)]

***Training set labels distribution***

In [ ]:
# Study labels distribution in training set 

training_labels = {
    'no_pain' : 0,
    'low_pain' : 0,
    'high_pain' : 0,
}


# Count occurrences of each activity for unique IDs in the training set
for id in y_train['sample_index'].unique():
    label = y_train[y_train['sample_index'] == id]['label'].values[0]
    training_labels[label] += 1

class_counts = torch.tensor([training_labels['no_pain'], training_labels['low_pain'], training_labels['high_pain']], dtype=torch.float)

# Print the distribution of training labels
print('Training labels:', training_labels)
print('Training distributions:\n')
for label in training_labels:
    stat = training_labels[label]/ (training_labels['no_pain'] + training_labels['low_pain'] + training_labels['high_pain'])
    print(f'{label}: {stat}')

In [ ]:
print(class_counts)

In [ ]:
print(len(class_counts))

***Validation set labels distribution***

In [ ]:
# Study labels distribution in validation set 

validation_labels = {
    'no_pain' : 0,
    'low_pain' : 0,
    'high_pain' : 0,
}


# Count occurrences of each activity for unique IDs in the validation set
for id in y_val['sample_index'].unique():
    label = y_val[y_val['sample_index'] == id]['label'].values[0]
    validation_labels[label] += 1

# Print the distribution of validation labels
print('Validation labels:', validation_labels)

# Print the distribution of validation labels
print('Validation labels:', validation_labels)
print('Validation distributions:\n')
for label in validation_labels:
    stat = validation_labels[label]/ (validation_labels['no_pain'] + validation_labels['low_pain'] + validation_labels['high_pain'])
    print(f'{label}: {stat}')

In [ ]:
# Mapping labels to integer data

label_map = {
    'no_pain' : 0,
    'low_pain' : 1,
    'high_pain' : 2
}

y_train['label'] = y_train['label'].map(label_map)
y_val['label'] = y_val['label'].map(label_map)

print(y_train[:10])
print(y_val[:10])

In [ ]:
y['label'] = y['label'].map(label_map)

In [ ]:
 # Window size and stride for sequence building

WINDOW_SIZE = 20 
STRIDE = 5 # the window need to be divisible by the stride

In [ ]:
data_cols = [col for col in X.columns if (not col == 'sample_index' and not col == 'time')]
print(data_cols)

In [ ]:
# Function to build sequences

def build_sequences(df, window, stride, isTest, data_cols):
    # Be sure that the window is divisible by the stride
    assert window%stride == 0

    #Lists to store sequences and corresponding labels
    dataset = []
    labels = []

    # Iterate over the ids
    for id in df['sample_index'].unique():
        # Extract data from the current id
        temp = df[df['sample_index'] == id][data_cols].values

        if(not isTest):
            # Retrive the label for the id
            label = y[y['sample_index'] == id]['label'].values[0]

        # Calculate padding length to ensure full windows
        padding_len = window - len(temp) % window

        # Create zero padding and concatenate with the data
        padding = np.zeros((padding_len, len(data_cols)), dtype='float32')
        temp = np.concatenate((temp, padding))

        # Build feature windows and associate them with labels
        idx = 0
        while idx + window <= len(temp):
            dataset.append(temp[idx:idx + window])
            if(not isTest):
                labels.append(label)
            idx += stride

    # Convert lists to numpy arrays for further processing
    dataset = np.array(dataset)
    if(not isTest):
        labels = np.array(labels)

    if(not isTest):
        return dataset, labels
    else:
        return dataset


In [ ]:
# Define the batch size, which is the number of samples in each batch
BATCH_SIZE = 512

In [ ]:
def make_loader(ds, batch_size, shuffle, drop_last):
    # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    # Create DataLoader with performance optimizations
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,  # Faster GPU transfer
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=4,  # Load 4 batches ahead
    )

# Training parameters

In [ ]:
# Cross-validation
K = 5                    # Number of splits (5 and 10 are considered good values)
TEST_SIZE = 0.2          # Validation/test proportion

# Training configuration
LEARNING_RATE = 1e-3
EPOCHS = 500
PATIENCE = 50

# Architecture
HIDDEN_LAYERS = 2        # Hidden layers
HIDDEN_SIZE = 128        # Neurons per layer

# Regularisation
DROPOUT_RATE = 0.2         # Dropout probability
L1_LAMBDA = 0          # L1 penalty
L2_LAMBDA = 0.0001     # L2 penalty

# Set up loss function and optimizer
# Training utilities
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
criterion = nn.CrossEntropyLoss()

In [ ]:
# Initialize best model tracking variables
best_model = None
best_performance = float('-inf')

# Embedding

In [ ]:
categorical_features = ['sample_index', 'time', 'n_legs', 'n_hands', 'n_eyes']

In [ ]:
embedding_map = {
    'two_legs' : 0,
    'two_hands' : 1,
    'two_eyes' : 2,
    'one+peg_leg' : 3,
    'one+hook_hand' : 4,
    'one+eye_patch' : 5
}

In [ ]:
def embeddingFeatures(X=X, categorical_fts=categorical_features, embedding_map = embedding_map, train_samples=train_samples, val_samples=val_samples):
    # Separate categorical data for future embedding
    X_cat = X[categorical_features]
    X_test_cat = X_test[categorical_features]

    # Delete of categorical features
    X_non_cat = X.drop(['n_legs', 'n_hands', 'n_eyes'], axis=1)
    X_test_non_cat = X_test.drop(['n_legs', 'n_hands', 'n_eyes'], axis=1)

    # Training and validation samples deifnition
    X_train_cat = X_cat[X_cat['sample_index'].isin(train_samples)]
    X_val_cat = X_cat[X_cat['sample_index'].isin(val_samples)]
    X_train_non_cat = X_non_cat[X_non_cat['sample_index'].isin(train_samples)]
    X_val_non_cat = X_non_cat[X_non_cat['sample_index'].isin(val_samples)]

    # Categorical values replacement - training set
    X_train_cat['n_legs'] = X_train_cat['n_legs'].replace('two', 'two_legs')
    X_train_cat['n_hands'] = X_train_cat['n_hands'].replace('two', 'two_hands')
    X_train_cat['n_eyes'] = X_train_cat['n_eyes'].replace('two', 'two_eyes')

    # Categorical values replacement - validation set
    X_val_cat['n_legs'] = X_val_cat['n_legs'].replace('two', 'two_legs')
    X_val_cat['n_hands'] = X_val_cat['n_hands'].replace('two', 'two_hands')
    X_val_cat['n_eyes'] = X_val_cat['n_eyes'].replace('two', 'two_eyes')

    # Categorical values replacement - test set
    X_test_cat['n_legs'] = X_test_cat['n_legs'].replace('two', 'two_legs')
    X_test_cat['n_hands'] = X_test_cat['n_hands'].replace('two', 'two_hands')
    X_test_cat['n_eyes'] = X_test_cat['n_eyes'].replace('two', 'two_eyes')

    #Embedding training
    X_train_cat['n_legs'] = X_train_cat['n_legs'].map(embedding_map)
    X_train_cat['n_hands'] = X_train_cat['n_hands'].map(embedding_map)
    X_train_cat['n_eyes'] = X_train_cat['n_eyes'].map(embedding_map)

    #Embedding validation
    X_val_cat['n_legs'] = X_val_cat['n_legs'].map(embedding_map)
    X_val_cat['n_hands'] = X_val_cat['n_hands'].map(embedding_map)
    X_val_cat['n_eyes'] = X_val_cat['n_eyes'].map(embedding_map)

    #Embedding test
    X_test_cat['n_legs'] = X_test_cat['n_legs'].map(embedding_map)
    X_test_cat['n_hands'] = X_test_cat['n_hands'].map(embedding_map)
    X_test_cat['n_eyes'] = X_test_cat['n_eyes'].map(embedding_map)

    return X_train_cat,  X_val_cat, X_train_non_cat, X_val_non_cat, X_test_cat, X_test_non_cat

In [ ]:
X_train_cat,  X_val_cat, X_train_non_cat, X_val_non_cat, X_test_cat, X_test_non_cat = embeddingFeatures()

***Normalization***

In [ ]:
joint_cols = [col for col in X_train.columns if re.match(r'joint_\d{2}', col)]

In [ ]:
#scaler = StandardScaler()
#scaler.fit(X_train_non_cat[joint_cols])

#X_train_non_cat[joint_cols] = scaler.transform(X_train_non_cat[joint_cols])
#X_val_non_cat[joint_cols] = scaler.transform(X_val_non_cat[joint_cols])
#X_test_non_cat[joint_cols] = scaler.transform(X_test_non_cat[joint_cols])

In [ ]:
def normalize(cols=joint_cols, X_train_non_cat=X_train_non_cat, X_val_non_cat=X_val_non_cat, X_test_non_cat=X_test_non_cat):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(X_train_non_cat[joint_cols])

    X_train_non_cat[joint_cols] = scaler.transform(X_train_non_cat[joint_cols])
    X_val_non_cat[joint_cols] = scaler.transform(X_val_non_cat[joint_cols])
    X_test_non_cat[joint_cols] = scaler.transform(X_test_non_cat[joint_cols])


In [ ]:
normalize(joint_cols, X_train_non_cat, X_val_non_cat, X_test_non_cat)

In [ ]:
# We remove 'joint_30' because .describe() shows its std. dev. is 0
data_cols_non_cat = [
    col for col in data_cols 
    if not col in ['n_legs', 'n_hands', 'n_eyes', 'joint_30']
]
print(f"List of {len(data_cols_non_cat)} non-categorical features (joint_30 removed):")
print(data_cols_non_cat)

In [ ]:
# This is your thresholding idea, implemented as a function.
# You can run this cell to denoise the data before building sequences.

def apply_thresholding(df, cols_to_threshold, threshold=1e-1):
    """Sets values in specific columns to 0 if they are below the threshold."""
    print(f"Applying threshold (< {threshold}) to {len(cols_to_threshold)} columns...")
    df_new = df.copy()
    for col in cols_to_threshold:
        df_new[col] = df_new[col].apply(lambda x: 0 if abs(x) < threshold else x)
    return df_new
    
# Define the columns you want to denoise (based on boxplots)
# These are the ones that look like noise stuck at zero
cols_to_denoise = ['joint_21', 'joint_22', 'joint_23', 'joint_24', 'joint_25']

# --- OPTIONAL: Run this to apply the denoising ---
print("--- Applying Threshold Denoising ---")
X_train_non_cat = apply_thresholding(X_train_non_cat, cols_to_denoise)
X_val_non_cat   = apply_thresholding(X_val_non_cat, cols_to_denoise)
X_test_non_cat  = apply_thresholding(X_test_non_cat, cols_to_denoise)

print("Uncomment the lines above in this cell to apply thresholding.")

In [ ]:
# --- START: Add Derivative Features (SKIPPED) ---
print("Original X_train_non_cat shape:", X_train_non_cat.shape)

# 1. Apply feature engineering
# --- DERIVATIVES SKIPPED ---
# print("Adding derivatives (SKIPPED)")
# X_train_non_cat = add_derivatives(X_train_non_cat, feature_cols=data_cols_non_cat)
# X_val_non_cat   = add_derivatives(X_val_non_cat, feature_cols=data_cols_non_cat)
# X_test_non_cat  = add_derivatives(X_test_non_cat, feature_cols=data_cols_non_cat)

# 2. Update the master list of features
# This list will now include only the original 34 features (35 - joint_30)
data_cols_non_cat = [col for col in X_train_non_cat.columns if col not in ['sample_index', 'time', 'joint_30']]

# 3. Store this count in a variable for the model
NUM_FEATURES = len(data_cols_non_cat)
print(f"Updated numerical feature count: {NUM_FEATURES}") # This should now print 34

# --- END: Add Derivative Features ---

In [ ]:
def build_sequences_cat(X_train_cat, X_val_cat, X_test_cat):
    
    X_train_seq_cat, y_train_seq_cat = build_sequences(X_train_cat, WINDOW_SIZE, STRIDE, False, ['n_legs', 'n_hands', 'n_eyes'])
    X_val_seq_cat, y_val_seq_cat = build_sequences(X_val_cat, WINDOW_SIZE, STRIDE, False, ['n_legs', 'n_hands', 'n_eyes'])
    X_test_seq_cat = build_sequences(X_test_cat, WINDOW_SIZE, STRIDE, True, ['n_legs', 'n_hands', 'n_eyes'])

    return X_train_seq_cat, y_train_seq_cat, X_val_seq_cat, y_val_seq_cat, X_test_seq_cat

In [ ]:
X_train_seq_cat, y_train_seq_cat, X_val_seq_cat, y_val_seq_cat, X_test_seq_cat = build_sequences_cat(X_train_cat, X_val_cat, X_test_cat)

In [ ]:
def build_sequences_not_cat(X_train_non_cat, X_val_non_cat, X_test_non_cat):

    X_train_seq_non_cat, y_train_seq_non_cat = build_sequences(X_train_non_cat, WINDOW_SIZE, STRIDE, False, data_cols_non_cat)
    X_val_seq_non_cat, y_val_seq_non_cat = build_sequences(X_val_non_cat, WINDOW_SIZE, STRIDE, False, data_cols_non_cat)
    X_test_seq_non_cat = build_sequences(X_test_non_cat, WINDOW_SIZE, STRIDE, True, data_cols_non_cat)
        
    return X_train_seq_non_cat, y_train_seq_non_cat, X_val_seq_non_cat, y_val_seq_non_cat, X_test_seq_non_cat

In [ ]:
X_train_seq_non_cat, y_train_seq_non_cat, X_val_seq_non_cat, y_val_seq_non_cat, X_test_seq_non_cat = build_sequences_not_cat(X_train_non_cat, X_val_non_cat, X_test_non_cat)

In [ ]:
num_classes = len(np.unique(y_train_seq_non_cat))
print(num_classes)

In [ ]:
X_train_seq_cat.shape, y_train_seq_cat.shape, X_val_seq_cat.shape, y_val_seq_cat.shape, X_test_seq_cat.shape

In [ ]:
X_train_seq_non_cat.shape, y_train_seq_non_cat.shape, X_val_seq_non_cat.shape, y_val_seq_non_cat.shape, X_test_seq_non_cat.shape

In [ ]:
train_ds_cat = TensorDataset(torch.from_numpy(X_train_seq_cat), torch.from_numpy(X_train_seq_non_cat), torch.from_numpy(y_train_seq_cat))
val_ds_cat   = TensorDataset(torch.from_numpy(X_val_seq_cat), torch.from_numpy(X_val_seq_non_cat) ,torch.from_numpy(y_val_seq_cat))
test_ds_cat = TensorDataset(torch.from_numpy(X_test_seq_cat), torch.from_numpy(X_test_seq_non_cat))

In [ ]:
train_cat_loader = make_loader(train_ds_cat, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_cat_loader   = make_loader(val_ds_cat, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_cat_loader   = make_loader(test_ds_cat, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [ ]:
for xb_cat, xb_non_cat, yb in train_cat_loader:
    print(type(xb_cat))
    print("categorical features batch shape:", xb_cat.shape)
    print("Numerical features batch shape:", xb_non_cat.shape)
    print("Label batch shape:", yb.shape)
    break

In [ ]:
for xb_cat, xb_non_cat in test_cat_loader:
    print(type(xb_cat))
    print("categorical features batch shape:", xb_cat.shape)
    print("Numerical features batch shape:", xb_non_cat.shape)
    break

In [ ]:
import torch
from torch import nn

class CnnLstmAttention(nn.Module):
    def __init__(
        self,
        hidden_size,
        num_layers,
        num_classes,
        rnn_type,
        bidirectional=False,
        dropout_rate=0.4,
        vocab=6,
        embedding_dim=3,
        numerical_feat=105  # This will be 105
    ):
        super().__init__()
        
        self.rnn_type = rnn_type
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional
        
        # --- Embedding Layer ---
        self.embedding = nn.Embedding(vocab, embedding_dim)
        # 3 features (legs, hands, eyes) * 3 dim each = 9
        embed_output_dim = int(embedding_dim * (vocab / 2)) 
        
        # --- 1D CNN Feature Extractor ---
        self.cnn_in_channels = numerical_feat # This is 105
        self.cnn_out_channels = 64  # Learn 64 new "pattern" features
        self.cnn_kernel_size = 3    # Look at 3 time steps at a time
        
        # This layer scans the raw sensor data
        self.cnn = nn.Conv1d(
            in_channels=self.cnn_in_channels,
            out_channels=self.cnn_out_channels,
            kernel_size=self.cnn_kernel_size,
            padding=1 # 'same' padding to keep seq_len=20
        )
        self.relu = nn.ReLU()

        # --- BiLSTM Layer ---
        # The LSTM input is the combined output of CNN (64) + Embeddings (9)
        rnn_input_dim = self.cnn_out_channels + embed_output_dim
        rnn_module = {'LSTM': nn.LSTM, 'GRU': nn.GRU}[rnn_type]
        
        self.rnn = rnn_module(
            input_size=rnn_input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=self.bidirectional,
            dropout=dropout_rate if num_layers > 1 else 0
        )
        
        # --- Tanh Attention Layer ---
        rnn_output_dim = hidden_size * (2 if bidirectional else 1)
        self.attn_projection = nn.Linear(rnn_output_dim, hidden_size)
        self.attn_score = nn.Linear(hidden_size, 1, bias=False)
        
        # --- Classifier ---
        self.classifier = nn.Linear(rnn_output_dim, num_classes)
        self.dropout = nn.Dropout(dropout_rate)

    def attention(self, rnn_output):
        # rnn_output shape: (batch, seq_len, rnn_output_dim)
        energy_hidden = torch.tanh(self.attn_projection(rnn_output))
        energy = self.attn_score(energy_hidden)
        weights = torch.nn.functional.softmax(energy, dim=1)
        context_vector = torch.sum(rnn_output * weights, dim=1)
        return context_vector, weights

    def forward(self, x_cat, x_non_cat):
        # x_cat shape: (batch, seq, 3)
        # x_non_cat shape: (batch, seq, 105)
        
        # 1. Embeddings
        # Output: (batch, seq, 9)
        x_cat_emb = self.embedding(x_cat.long())
        x_cat_emb = x_cat_emb.view(x_cat_emb.size(0), x_cat_emb.size(1), -1)
        
        # 2. CNN on Numerical Data
        # CNN needs (batch, channels, seq_len)
        # Input: (batch, 105, 20)
        x_non_cat_cnn = x_non_cat.permute(0, 2, 1) 
        x_cnn_out = self.relu(self.cnn(x_non_cat_cnn))
        # Convert back to (batch, seq_len, channels)
        # Output: (batch, 20, 64)
        x_cnn_out = x_cnn_out.permute(0, 2, 1)
        
        # 3. Concatenate CNN features and Embeddings
        # Output: (batch, 20, 73)
        x = torch.cat((x_cnn_out, x_cat_emb), dim=-1)
        
        # 4. BiLSTM
        rnn_out, _ = self.rnn(x)
        
        # 5. Attention
        context_vector, _ = self.attention(rnn_out)
        
        # 6. Classifier
        context_vector = self.dropout(context_vector)
        logits = self.classifier(context_vector)
        
        return logits

In [ ]:
def train_one_epoch_cat(model, train_loader, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0):
  
    model.train()  # Set model to training mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Iterate through training batches
    for batch_idx, (inputs_cat, inputs_non_cat, targets) in enumerate(train_loader):
        # Move data to device (GPU/CPU)
        inputs_cat, inputs_non_cat, targets = inputs_cat.to(device).float(), inputs_non_cat.to(device).float(), targets.to(device)

        # Clear gradients from previous step
        optimizer.zero_grad(set_to_none=True)

        # Forward pass with mixed precision (if CUDA available)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs_cat, inputs_non_cat)
            loss = criterion(logits, targets)


        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Accumulate metrics
        running_loss += loss.item() * inputs_cat.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_f1

In [ ]:
def fit_cat(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device,
        l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_f1", mode='max',
        restore_best_weights=True, writer=None, verbose=10, experiment_name=""):
  

    # Initialize metrics tracking
    training_history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [], 'val_f1': []
    }

    # Configure early stopping if patience is set
    if patience > 0:
        patience_counter = 0
        best_metric = float('-inf') if mode == 'max' else float('inf')
        best_epoch = 0

    print(f"Training {epochs} epochs...")

    # Main training loop: iterate through epochs
    for epoch in range(1, epochs + 1):

        # Forward pass through training data, compute gradients, update weights
        train_loss, train_f1 = train_one_epoch_cat(
            model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
        )

        # Evaluate model on validation data without updating weights
        val_loss, val_f1 = validate_one_epoch_cat(
            model, val_loader, criterion, device
        )

        # Store metrics for plotting and analysis
        training_history['train_loss'].append(train_loss)
        training_history['val_loss'].append(val_loss)
        training_history['train_f1'].append(train_f1)
        training_history['val_f1'].append(val_f1)

        # Write metrics to TensorBoard for visualization
        if writer is not None:
            log_metrics_to_tensorboard(
                writer, epoch, train_loss, train_f1, val_loss, val_f1, model
            )

        # Print progress every N epochs or on first epoch
        if verbose > 0:
            if epoch % verbose == 0 or epoch == 1:
                print(f"Epoch {epoch:3d}/{epochs} | "
                    f"Train: Loss={train_loss:.4f}, F1 Score={train_f1:.4f} | "
                    f"Val: Loss={val_loss:.4f}, F1 Score={val_f1:.4f}")

        # Early stopping logic: monitor metric and save best model
        if patience > 0:
            current_metric = training_history[evaluation_metric][-1]
            is_improvement = (current_metric > best_metric) if mode == 'max' else (current_metric < best_metric)

            if is_improvement:
                best_metric = current_metric
                best_epoch = epoch
                torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping triggered after {epoch} epochs.")
                    break

    # Restore best model weights if early stopping was used
    if restore_best_weights and patience > 0:
        model.load_state_dict(torch.load("models/"+experiment_name+'_model.pt'))
        print(f"Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}")

    # Save final model if no early stopping
    if patience == 0:
        torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')

    # Close TensorBoard writer
    if writer is not None:
        writer.close()

    return model, training_history

In [ ]:
def validate_one_epoch_cat(model, val_loader, criterion, device):

    model.eval()  # Set model to evaluation mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Disable gradient computation for validation
    with torch.no_grad():
        for inputs_cat, inputs_non_cat, targets in val_loader:
            # Move data to device
            inputs_cat, inputs_non_cat, targets = inputs_cat.to(device).float(), inputs_non_cat.to(device).float(), targets.to(device)

            # Forward pass with mixed precision (if CUDA available)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs_cat, inputs_non_cat)
                loss = criterion(logits, targets)

            # Accumulate metrics
            running_loss += loss.item() * inputs_cat.size(0)
            predictions = logits.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_accuracy = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_accuracy

## Model: CNN-BiLSTM-Attention with Weighted Loss


In [ ]:
# --- Define Weighted Loss ---
# This is the same logic as before, ensuring we use weighted loss for all tuning
weights = 1/class_counts  # To give more importance to misclassifications of rare classes
weights = weights/weights.mean() # weights normalizations
weights = weights.to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

print("Weighted criterion set.")

## Final Training Run


In [ ]:
train_samples = samples

In [ ]:
# --- Final Model Initialization & Training (with 35 Features) ---

# 1. Define your BEST Hyperparameters
BEST_LR = 0.001
BEST_L2 = 0.0001        # <--- Reverted to original
BEST_DROPOUT = 0.4      # <--- Reverted to original
BEST_BATCH_SIZE = 512

# 2. Re-create loaders with the correct batch size
print(f"Recreating loaders with Batch Size: {BEST_BATCH_SIZE}")
train_cat_loader = make_loader(train_ds_cat, batch_size=BEST_BATCH_SIZE, shuffle=True, drop_last=False)
val_cat_loader   = make_loader(val_ds_cat, batch_size=BEST_BATCH_SIZE, shuffle=False, drop_last=False)
test_cat_loader  = make_loader(test_ds_cat, batch_size=BEST_BATCH_SIZE, shuffle=False, drop_last=False)

# 3. Instantiate the NEW CNN-LSTM Attention Model
#    This will now receive numerical_feat=35
lstm_bidir_model_emb_weighted = CnnLstmAttention(
    hidden_size=HIDDEN_SIZE,
    num_layers=HIDDEN_LAYERS,
    num_classes=num_classes,
    dropout_rate=BEST_DROPOUT,
    bidirectional=True,
    rnn_type='LSTM',
    embedding_dim=3,
    numerical_feat=NUM_FEATURES  # This will be 35
).to(device)

# 4. Create a NEW optimizer for the NEW model's parameters
optimizer = torch.optim.AdamW(
    lstm_bidir_model_emb_weighted.parameters(), 
    lr=BEST_LR, 
    weight_decay=BEST_L2
)

print(f"CNN-LSTM-Attention Model initialized with {NUM_FEATURES} features and Dropout={BEST_DROPOUT}")
print(f"NEW optimizer created with LR={BEST_LR} and L2={BEST_L2}")
print("--- Starting Final Training ---")

# 5. Run the Training
lstm_bidir_model_emb_weighted, training_history = fit_cat(
    model=lstm_bidir_model_emb_weighted,
    train_loader=train_cat_loader,
    val_loader=val_cat_loader,
    epochs=EPOCHS,
    criterion=criterion,
    optimizer=optimizer,
    scaler=scaler,
    device=device,
    verbose=10,
    experiment_name="cnn_lstm_attn_original_35_features", # New experiment name
    patience=PATIENCE
)

# Evaluation

In [ ]:
print(len(test_cat_loader))

In [ ]:
# --- Corrected Evaluation Cell ---

# 1. Set model to evaluation mode
lstm_bidir_model_emb_weighted.eval()

# 2. Lists to store predictions
sample_indices = []
predictions = []

# 3. Mapping from class index to label name
idx_to_label = {0: 'no_pain', 1: 'low_pain', 2: 'high_pain'}

# --- FIX 1: Use the correct BATCH_SIZE ---
# Use the *exact* batch size variable that was used to create the test_loader
# This variable is set in your training cell.
try:
    loader_batch_size = BEST_BATCH_SIZE
    print(f"Using batch size from training cell: {loader_batch_size}")
except NameError:
    print(f"Warning: BEST_BATCH_SIZE not found. Falling back to BATCH_SIZE.")
    loader_batch_size = BATCH_SIZE

# 4. Generate predictions
with torch.no_grad():
    sample_count = 0
    
    for batch_idx, (batch_cat, batch_non_cat) in enumerate(test_cat_loader):
        
        # (Your logic for handling list/tensor is fine)
        if isinstance(batch_cat, list):
            xb_cat = batch_cat[0]
        else:
            xb_cat = batch_cat
        if isinstance(batch_non_cat, list):
            xb_non_cat = batch_non_cat[0]
        else:
            xb_non_cat = batch_non_cat
            
        xb_cat = xb_cat.to(device).float()
        xb_non_cat = xb_non_cat.to(device).float()

        # Get model predictions
        logits = lstm_bidir_model_emb_weighted(xb_cat, xb_non_cat)
        preds = logits.argmax(dim=1).cpu().numpy()

        # Convert predictions to labels and store
        for i, pred_idx in enumerate(preds):
            
            # --- FIX 1 (Continued): Use the correct batch size in logic ---
            # This calculates the window's "global index"
            global_window_index = (batch_idx * loader_batch_size) + i
            
            # We know there are 33 windows per sample
            if (global_window_index % 33 == 0):
                sample_indices.append(f"{sample_count:03d}")
                predictions.append(idx_to_label[pred_idx])
                sample_count += 1

# 5. Create DataFrame
submission_df = pd.DataFrame({
    'sample_index': sample_indices,
    'label': predictions
})

# --- FIX 2: Use the experiment_name for the filename ---
# This variable is set in your training cell.
if 'experiment_name' not in globals():
    experiment_name = "my_default_submission"

experiment_name = "cnn_lstm_attn_thresh" # <-- Use the name from cell 114
submission_filename = f"{experiment_name}.csv"

# 6. Save to a UNIQUE CSV file
submission_df.to_csv(submission_filename, index=False)

print(f"--- Evaluation Complete ---")
print(f"Submission file saved as: {submission_filename}")
print(f"Total samples predicted: {len(submission_df)}")
print("\nPrediction distribution:")
print(submission_df['label'].value_counts())
print("\nSubmission Head:")
print(submission_df.head())

In [ ]:
print(submission_df.head(10))

In [ ]:
print("\nPrediction distribution:")
print(submission_df['label'].value_counts())